# Feature Engineering Analysis
Detailed analysis of engineered features across all data modalities.

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# PySpark imports
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Data analysis & visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Project imports
from src.data_processing.spark_session import create_spark_session
from src.data_processing.data_loader import HealthcareDataLoader
from src.feature_engineering.structured_features import StructuredFeatureEngineer
from src.feature_engineering.nlp_features import NLPFeatureEngineer
from src.feature_engineering.symptom_features import SymptomFeatureEngineer
from src.feature_engineering.lab_features import LabFeatureEngineer

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Create Spark session and load data
spark = create_spark_session(app_name="FeatureEngineering", master="local[*]")
loader = HealthcareDataLoader(spark, data_dir=os.path.join(project_root, 'data'))

patients_df = loader.load_patients()
vitals_df = loader.load_vitals()
symptoms_df = loader.load_symptoms()
lab_results_df = loader.load_lab_results()
clinical_notes_df = loader.load_clinical_notes()
ground_truth_df = loader.load_ground_truth()

ground_truth_pd = ground_truth_df.toPandas()
print("Data loaded successfully!")

## Structured Feature Engineering

In [ ]:
# Compute clinical severity scores using StructuredFeatureEngineer
struct_engineer = StructuredFeatureEngineer(spark)

# Engineer structured features (MEWS, qSOFA, Charlson Comorbidity Index)
structured_features_df = struct_engineer.engineer_features(patients_df, vitals_df)
structured_pd = structured_features_df.toPandas()
structured_merged = structured_pd.merge(ground_truth_pd[['patient_id', 'diagnosis', 'risk_level']], 
                                         on='patient_id', how='left')

# Show distributions of clinical scores by diagnosis
score_cols = [c for c in structured_merged.columns if any(s in c.lower() for s in ['mews', 'qsofa', 'charlson', 'score', 'severity'])]
print(f"Engineered structured features: {list(structured_merged.columns)}")
print(f"\nClinical score columns detected: {score_cols}")

# Display summary statistics per diagnosis
for col in score_cols[:5]:
    print(f"\n--- {col} ---")
    print(structured_merged.groupby('diagnosis')[col].describe().round(2))

In [ ]:
# Plot MEWS score distribution per diagnosis class (box plots)
fig, axes = plt.subplots(1, min(len(score_cols), 3), figsize=(6 * min(len(score_cols), 3), 6))
if len(score_cols) == 1:
    axes = [axes]

for idx, col in enumerate(score_cols[:3]):
    diagnoses = sorted(structured_merged['diagnosis'].dropna().unique())
    data_groups = [structured_merged[structured_merged['diagnosis'] == d][col].dropna().values 
                   for d in diagnoses]
    
    bp = axes[idx].boxplot(data_groups, labels=diagnoses, patch_artist=True)
    colors = plt.cm.Set2(np.linspace(0, 1, len(diagnoses)))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    axes[idx].set_title(f'{col} by Diagnosis', fontsize=13, fontweight='bold')
    axes[idx].set_ylabel(col)
    axes[idx].tick_params(axis='x', rotation=45)

plt.suptitle('Clinical Severity Scores Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'outputs', 'severity_scores.png'), dpi=150, bbox_inches='tight')
plt.show()

## Symptom Cluster Analysis

In [ ]:
# Compute symptom clusters and create radar/spider chart
symptom_engineer = SymptomFeatureEngineer(spark)
symptom_features_df = symptom_engineer.engineer_features(symptoms_df)
symptom_pd = symptom_features_df.toPandas()
symptom_merged = symptom_pd.merge(ground_truth_pd[['patient_id', 'diagnosis']], on='patient_id', how='left')

# Identify cluster columns
cluster_cols = [c for c in symptom_merged.columns 
                if any(s in c.lower() for s in ['cluster', 'respiratory', 'cardiac', 'neuro', 'gastro', 
                                                   'metabolic', 'infectious', 'symptom_count', 'severity'])
                and c not in ['patient_id', 'diagnosis']]

if not cluster_cols:
    # Use all numeric symptom feature columns
    cluster_cols = [c for c in symptom_merged.select_dtypes(include=[np.number]).columns 
                    if c not in ['patient_id']]

print(f"Symptom feature columns: {cluster_cols[:15]}")

# Radar/spider chart of mean cluster values per diagnosis
diagnoses = sorted(symptom_merged['diagnosis'].dropna().unique())
display_cols = cluster_cols[:8]  # Limit for readability

if len(display_cols) >= 3:
    fig = go.Figure()
    colors = px.colors.qualitative.Set2
    
    for i, diag in enumerate(diagnoses):
        diag_data = symptom_merged[symptom_merged['diagnosis'] == diag]
        mean_vals = [diag_data[col].mean() for col in display_cols]
        
        # Normalize to 0-1 scale for radar chart
        max_vals = [symptom_merged[col].max() for col in display_cols]
        norm_vals = [m / mx if mx > 0 else 0 for m, mx in zip(mean_vals, max_vals)]
        norm_vals.append(norm_vals[0])  # Close the polygon
        
        fig.add_trace(go.Scatterpolar(
            r=norm_vals,
            theta=display_cols + [display_cols[0]],
            fill='toself',
            name=diag,
            line=dict(color=colors[i % len(colors)])
        ))
    
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        showlegend=True,
        title=dict(text='Symptom Cluster Profiles by Diagnosis (Normalized)', font=dict(size=16)),
        width=800, height=600
    )
    fig.show()
else:
    print("Insufficient cluster features for radar chart. Showing bar plot instead.")
    means = symptom_merged.groupby('diagnosis')[cluster_cols].mean()
    means.plot(kind='bar', figsize=(12, 6))
    plt.title('Mean Symptom Features by Diagnosis')
    plt.tight_layout()
    plt.show()

## Lab Feature Engineering

In [ ]:
# Compute lab deviation scores and organ panel scores
lab_engineer = LabFeatureEngineer(spark)
lab_features_df = lab_engineer.engineer_features(lab_results_df)
lab_feat_pd = lab_features_df.toPandas()
lab_feat_merged = lab_feat_pd.merge(ground_truth_pd[['patient_id', 'diagnosis']], on='patient_id', how='left')

# Identify organ panel score columns
panel_cols = [c for c in lab_feat_merged.columns 
              if any(s in c.lower() for s in ['liver', 'renal', 'metabolic', 'cbc', 'panel', 
                                                 'deviation', 'abnormal', 'composite'])]

if not panel_cols:
    panel_cols = [c for c in lab_feat_merged.select_dtypes(include=[np.number]).columns 
                  if c not in ['patient_id']][:8]

print(f"Lab feature columns: {panel_cols}")

# Plot organ panel scores by diagnosis (grouped bar chart)
diagnoses = sorted(lab_feat_merged['diagnosis'].dropna().unique())
display_panels = panel_cols[:6]

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(diagnoses))
width = 0.8 / max(len(display_panels), 1)

for i, panel in enumerate(display_panels):
    means = [lab_feat_merged[lab_feat_merged['diagnosis'] == d][panel].mean() for d in diagnoses]
    ax.bar(x + i * width, means, width, label=panel, alpha=0.85)

ax.set_xlabel('Diagnosis', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Lab Panel Scores by Diagnosis', fontsize=16, fontweight='bold')
ax.set_xticks(x + width * len(display_panels) / 2)
ax.set_xticklabels(diagnoses, rotation=45, ha='right')
ax.legend(title='Panel', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'outputs', 'lab_panel_scores.png'), dpi=150, bbox_inches='tight')
plt.show()

## NLP Feature Engineering

In [ ]:
# Extract TF-IDF features using NLPFeatureEngineer
nlp_engineer = NLPFeatureEngineer(spark)
nlp_features_df = nlp_engineer.engineer_features(clinical_notes_df)
nlp_pd = nlp_features_df.toPandas()
nlp_merged = nlp_pd.merge(ground_truth_pd[['patient_id', 'diagnosis']], on='patient_id', how='left')

# Show top TF-IDF terms per diagnosis class
print("NLP Feature columns:", list(nlp_merged.columns[:20]))

# Identify TF-IDF or text feature columns
tfidf_cols = [c for c in nlp_merged.columns if 'tfidf' in c.lower() or 'term' in c.lower()]
if not tfidf_cols:
    tfidf_cols = [c for c in nlp_merged.select_dtypes(include=[np.number]).columns 
                  if c not in ['patient_id']]

# Show top terms (if vocabulary is accessible)
if hasattr(nlp_engineer, 'vocabulary') or hasattr(nlp_engineer, 'get_vocabulary'):
    try:
        vocab = nlp_engineer.vocabulary if hasattr(nlp_engineer, 'vocabulary') else nlp_engineer.get_vocabulary()
        print(f"\nVocabulary size: {len(vocab)}")
    except:
        print("\nVocabulary not directly accessible")

print(f"\nTotal NLP features: {len(tfidf_cols)}")
print(f"Sample features: {tfidf_cols[:10]}")

In [ ]:
# Extract medical entities and show entity count distributions
entity_cols = [c for c in nlp_merged.columns if any(s in c.lower() for s in ['entity', 'ner', 'medical', 'drug', 'condition', 'procedure'])]

if entity_cols:
    fig, ax = plt.subplots(figsize=(12, 6))
    entity_means = nlp_merged.groupby('diagnosis')[entity_cols].mean()
    entity_means.plot(kind='bar', ax=ax, width=0.8)
    ax.set_title('Medical Entity Counts by Diagnosis', fontsize=14, fontweight='bold')
    ax.set_xlabel('Diagnosis')
    ax.set_ylabel('Mean Count')
    ax.legend(title='Entity Type', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
else:
    # Show NLP feature distributions as alternative
    display_feats = tfidf_cols[:6]
    if display_feats:
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        axes = axes.flatten()
        for idx, col in enumerate(display_feats):
            for diag in sorted(nlp_merged['diagnosis'].dropna().unique()):
                data = nlp_merged[nlp_merged['diagnosis'] == diag][col].dropna()
                axes[idx].hist(data, bins=20, alpha=0.5, label=diag)
            axes[idx].set_title(col, fontsize=11)
            axes[idx].legend(fontsize=7)
        plt.suptitle('NLP Feature Distributions by Diagnosis', fontsize=15, fontweight='bold')
        plt.tight_layout()
        plt.show()

## Feature Importance Preview

In [ ]:
# Feature importance using mutual information
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

# Combine all engineered features
all_features = structured_merged.merge(symptom_merged, on='patient_id', how='outer', suffixes=('', '_sym'))
all_features = all_features.merge(lab_feat_merged, on='patient_id', how='outer', suffixes=('', '_lab'))

# Get diagnosis column
if 'diagnosis' in all_features.columns:
    diag_col = 'diagnosis'
elif 'diagnosis_sym' in all_features.columns:
    diag_col = 'diagnosis_sym'
else:
    diag_col = [c for c in all_features.columns if 'diagnosis' in c][0]

# Select numeric feature columns
feature_cols = [c for c in all_features.select_dtypes(include=[np.number]).columns 
                if c not in ['patient_id']]

# Prepare data
X = all_features[feature_cols].fillna(0)
y = all_features[diag_col].fillna('unknown')
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Compute mutual information
mi_scores = mutual_info_classif(X, y_encoded, random_state=42)
mi_df = pd.DataFrame({'feature': feature_cols, 'mi_score': mi_scores})
mi_df = mi_df.sort_values('mi_score', ascending=False)

# Plot top-30 features
top_n = min(30, len(mi_df))
fig, ax = plt.subplots(figsize=(10, 12))
ax.barh(range(top_n), mi_df['mi_score'].values[:top_n], color='steelblue', edgecolor='black')
ax.set_yticks(range(top_n))
ax.set_yticklabels(mi_df['feature'].values[:top_n])
ax.invert_yaxis()
ax.set_title(f'Top {top_n} Features by Mutual Information', fontsize=16, fontweight='bold')
ax.set_xlabel('Mutual Information Score')
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'outputs', 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTotal features analyzed: {len(feature_cols)}")
print(f"\nTop 10 most informative features:")
for _, row in mi_df.head(10).iterrows():
    print(f"  {row['feature']}: {row['mi_score']:.4f}")

## Dimensionality Reduction Visualization

In [ ]:
# PCA and t-SNE on combined features
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# PCA - 2D projection
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

diagnoses = all_features[diag_col].fillna('unknown').values
unique_diags = sorted(set(diagnoses))
colors = plt.cm.Set2(np.linspace(0, 1, len(unique_diags)))

for i, diag in enumerate(unique_diags):
    mask = diagnoses == diag
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=[colors[i]], label=diag, alpha=0.6, s=30)

axes[0].set_title(f'PCA Projection (Explained Var: {pca.explained_variance_ratio_.sum():.1%})', 
                  fontsize=14, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[0].legend(fontsize=9, markerscale=1.5)

# t-SNE - 2D projection
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(X_scaled) - 1), n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

for i, diag in enumerate(unique_diags):
    mask = diagnoses == diag
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=[colors[i]], label=diag, alpha=0.6, s=30)

axes[1].set_title('t-SNE Projection', fontsize=14, fontweight='bold')
axes[1].set_xlabel('t-SNE Dimension 1')
axes[1].set_ylabel('t-SNE Dimension 2')
axes[1].legend(fontsize=9, markerscale=1.5)

plt.suptitle('Dimensionality Reduction of Combined Features', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'outputs', 'dimensionality_reduction.png'), dpi=150, bbox_inches='tight')
plt.show()

# PCA explained variance
print(f"\nPCA Analysis:")
print(f"  Components needed for 95% variance: {np.argmax(np.cumsum(PCA(random_state=42).fit(X_scaled).explained_variance_ratio_) >= 0.95) + 1}")
print(f"  Top 2 components explain: {pca.explained_variance_ratio_.sum():.1%}")

# Stop Spark session
spark.stop()
print("\nSpark session stopped. Feature engineering analysis complete!")